# Profiling execution times

In [ ]:
import pprint
from time import sleep

from ufl.core.expr import Expr
from ufl import Form, inner, grad, dx, TestFunction, TrialFunction
from lucifex.mesh import rectangle_mesh
from lucifex.fem import Function, Constant
from lucifex.solver import bvp, projection, interpolation
from lucifex.utils.py_utils import log_timing
from lucifex.plt import plot_bar
from lucifex.pde.diffusion import poisson

def sleepy_multiply(a, b, t):
    sleep(t)
    return a * b

texec = {}
log_timing(sleep, texec, n=3)(0.123)
a_times_b = log_timing(sleepy_multiply, texec, n=3)(3, 4, 0.321)

print(f'a * b = {a_times_b}')
pprint.pp(texec)

## increasing mesh resolution

In [ ]:
texec_poisson = {}

n_solve = 3
Nx = [int(2 ** i) for i in range(7)]
for nx in Nx:
    mesh = rectangle_mesh(1.0, 1.0, nx, nx)
    u = Function((mesh, 'P', 1))
    f = Constant(mesh, 1.0)
    u_solver = bvp(poisson, overwrite=True)(u, f)
    key = f'poisson_Nx={nx}'
    log_timing(u_solver.solve, texec_poisson, key, n=n_solve)()

In [ ]:
fig, ax = plot_bar((texec_poisson.values()), Nx, x_label='$N_x$', y_label='$t_{\mathrm{exec}}$')
ax.set_yscale('log')

## caching assembled matrices

In [ ]:
texec_caching = {}

n_solve = 3
nx = 100
mesh = rectangle_mesh(1.0, 1.0, nx, nx)
cache_matrix = (False, True)
for cm in cache_matrix:
    u = Function((mesh, 'P', 1))
    f = Constant(mesh, 1.0)
    u_solver = bvp(poisson, overwrite=True, cache_matrix=cm)(u, f)
    key = f'poisson_Nx={nx}_cache={cm}'
    log_timing(u_solver.solve, texec_caching, key, n=n_solve)()
    print(
        f'cache_matrix={cm}', 
        f"assembly_count={u_solver.get_matrix().getAttr('assembly_count')}",
    )


In [ ]:
plot_bar((texec_caching.values()), cache_matrix, y_label='$t_{\mathrm{exec}}$')

## projection versus interpolation

In [ ]:
def exponentiate(a, b):
    return a ** b

texec_expn = {}

n_solve = 5
nx = 100
mesh = rectangle_mesh(1.0, 1.0, nx, nx)
a = Function((mesh, 'P', 1), lambda x: x[0] * x[1])
b = Function((mesh, 'P', 1), lambda x: x[0])

methods = ('projection', 'interpolation')
for mthd in methods:
    u = Function((mesh, 'P', 1))
    if mthd == 'projection':
        ab_solver = projection(u, exponentiate, overwrite=True)(a, b)
    elif mthd == 'interpolation':
        ab_solver = interpolation(u, exponentiate, overwrite=True)(a, b)
    else:
        raise ValueError
    log_timing(ab_solver.solve, texec_expn, mthd, n=n_solve)()
    
pprint.pp(texec_expn)

In [ ]:
plot_bar((texec_expn.values()), methods, y_label='$t_{\mathrm{exec}}$')